# Forecasting Modeling

This notebook trains baseline and machine-learning demand forecasts, then converts the predictions into a business-facing replenishment ranking.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'src').exists() and (candidate / 'scripts').exists():
        project_root = candidate
        break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.forecasting.baseline import generate_baselines
from src.forecasting.evaluation import evaluate_forecasts, generate_replenishment_recommendations
from src.forecasting.features import create_forecasting_features
from src.forecasting.model import train_random_forest_model
from src.forecasting.preprocessing import build_forecasting_base

build_forecasting_base()
create_forecasting_features()
train_random_forest_model()

baseline_output = generate_baselines()
metrics = evaluate_forecasts()
replenishment = generate_replenishment_recommendations()

print('Baseline sample:')
print(baseline_output.head())
print('\nForecast metrics:')
print(metrics)
print('\nTop replenishment priority items:')
print(replenishment.sort_values('forecasted_demand_28d', ascending=False).head(10))


Baseline sample:
        date store_id   item_id  naive_forecast  seasonal_naive_forecast
0 2023-06-30     CA_1  ITEM_001            51.0                     44.0
1 2023-07-01     CA_1  ITEM_001            51.0                     44.0
2 2023-07-02     CA_1  ITEM_001            51.0                     44.0
3 2023-07-03     CA_1  ITEM_001            51.0                     44.0
4 2023-07-04     CA_1  ITEM_001            51.0                     44.0

Forecast metrics:
            model     MAE    RMSE     MAPE
0           Naive  2.2723  2.8743   5.4368
1  Seasonal Naive  5.1959  6.0363  12.1581
2   Random Forest  0.7251  0.9444   1.7434

Top replenishment priority items:
     item_id store_id  forecasted_demand_7d  forecasted_demand_28d  \
47  ITEM_024     TX_1                 302.0                 1255.0   
51  ITEM_004     WI_1                 300.0                 1247.0   
66  ITEM_019     WI_1                 296.0                 1239.0   
18  ITEM_019     CA_1                 2